In [105]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [106]:
df = pd.read_stata("../data/data.dta")

C:\Users\mdmah\AppData\Local\Temp\ipykernel_36168\1530219147.py:1: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df = pd.read_stata("../data/data.dta")


In [107]:
cols = [
    "caseid_new",

    # identity / grouping
    "w1_identity_all_modified",

    # surveyed flags
    "w2_surveyed",
    "w3_surveyed",

    # weights
    "w3_attrition_adj_weight",

    # relationship quality
    "w1_q34",
    "w2_rel_qual_combo",
    "w3_rel_qual",

    # sex frequency
    "w1_sex_frequency",
    "w2_sex_frequency",
    "w3_sex_frequency",

    # fights
    "w2_fight",
    "w3_fight",

    # flirting
    "w2_flirt",
    "w3_flirt",

    # relatives
    "w2_relatives",
    "w3_relatives",

    # monogamy
    "w2_p_monogamy",
    "w3_p_monogamy",

    # optional contextual variable if you want later analysis
    "w2_coronavirus_effect_combo"
]

In [108]:
df_time = df[cols].copy()
df_time = df_time[
    (df_time["w2_surveyed"] == "yes") &
    (df_time["w3_surveyed"] == "yes")
].copy()

print("Longitudinal sample shape:", df_time.shape)

Longitudinal sample shape: (1605, 20)


In [109]:
def classify_orientation(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower()
    if any(i in x for i in ["gay", "lesbian", "bisexual"]):
        return "LGB"
    elif "straight" in x or "hetero" in x:
        return "Straight"
    else:
        return np.nan

df_time["group"] = df_time["w1_identity_all_modified"].apply(classify_orientation)
df_time = df_time.dropna(subset=["group"]).copy()

print("\nGroup counts:")
print(df_time["group"].value_counts(dropna=False))


Group counts:
group
Straight    1356
LGB          239
Name: count, dtype: int64


In [ ]:
quality_map = {
    "Excellent": 5,
    "Good": 4,
    "Fair": 3,
    "Poor": 2,
    "Very Poor": 1
}

df_time["q_w1"] = df_time["w1_q34"].map(quality_map)
df_time["q_w2"] = df_time["w2_rel_qual_combo"].map(quality_map)
df_time["q_w3"] = df_time["w3_rel_qual"].map(quality_map)

df_time = df_time.dropna(subset=["q_w1", "q_w2", "q_w3"]).copy()

df_time = df_time.dropna(subset=["q_w1", "q_w2", "q_w3"]).copy()

for col in ["q_w1", "q_w2", "q_w3"]:
    df_time[col] = pd.to_numeric(df_time[col], errors="coerce")


In [111]:
print(df_time["w1_q34"].value_counts(dropna=False))
print(df_time["w2_rel_qual_combo"].value_counts(dropna=False))
print(df_time["w3_rel_qual"].value_counts(dropna=False))

w1_q34
Excellent    788
Good         397
NaN          304
Fair          87
Poor          13
Very Poor      6
Name: count, dtype: int64
w2_rel_qual_combo
Excellent    691
Good         441
NaN          347
Fair          83
Poor          25
Very Poor      8
Name: count, dtype: int64
w3_rel_qual
Excellent    635
Good         479
NaN          354
Fair          94
Poor          27
Very Poor      6
Name: count, dtype: int64


In [112]:
numeric_cols = [
    "w3_attrition_adj_weight",
    "q_w1", "q_w2", "q_w3",
]

for col in numeric_cols:
    df_time[col] = pd.to_numeric(df_time[col], errors="coerce")

# keep valid weights
df_time = df_time.dropna(subset=["w3_attrition_adj_weight"]).copy()
df_time = df_time[df_time["w3_attrition_adj_weight"] > 0].copy()

print("\nWeight summary:")
print(df_time["w3_attrition_adj_weight"].describe())


Weight summary:
count    1595.000000
mean        0.959825
std         0.979318
min         0.026903
25%         0.429797
50%         0.687882
75%         1.117998
max        10.109859
Name: w3_attrition_adj_weight, dtype: float64


In [113]:
def weighted_mean(x, w):
    x = pd.to_numeric(x, errors="coerce")
    w = pd.to_numeric(w, errors="coerce")
    mask = x.notna() & w.notna() & (w > 0)
    x = x[mask]
    w = w[mask]
    return np.average(x, weights=w) if len(x) > 0 else np.nan

group_means_weighted = df_time.groupby("group", observed=True).apply(
    lambda g: pd.Series({
        "w1": weighted_mean(g["q_w1"], g["w3_attrition_adj_weight"]),
        "w2": weighted_mean(g["q_w2"], g["w3_attrition_adj_weight"]),
        "w3": weighted_mean(g["q_w3"], g["w3_attrition_adj_weight"]),
    })
)

print(group_means_weighted)

                w1        w2        w3
group                                 
LGB       4.289384  4.220866  4.229963
Straight  4.462536  4.425212  4.364380


C:\Users\mdmah\AppData\Local\Temp\ipykernel_36168\3096799242.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  group_means_weighted = df_time.groupby("group", observed=True).apply(


In [114]:
df_time["q_w1"] = pd.to_numeric(df_time["q_w1"])
df_time["q_w2"] = pd.to_numeric(df_time["q_w2"])
df_time["q_w3"] = pd.to_numeric(df_time["q_w3"])

group_means = df_time.groupby("group")[["q_w1", "q_w2", "q_w3"]].mean()
print(group_means)

              q_w1      q_w2      q_w3
group                                 
LGB       4.363636  4.245161  4.169811
Straight  4.530195  4.453797  4.408503
